# Notebook A — LoRA fine-tune (Kaggle, self-contained)

Clones the repo from GitHub, materialises crops on Kaggle, trains a LoRA adapter on Qwen2.5-VL-7B (4-bit Unsloth), pushes the adapter to HuggingFace Hub.

**One-time setup (outside the notebook):**
1. Push this repo to GitHub (already done: `dhodyrev/handwritten-to-data`).
2. Create an empty HF model repo at `dkhodyriev/htr-qwen25vl-transcribe-lora` (private OK).
3. Generate an HF write token at https://huggingface.co/settings/tokens. Kaggle → Add-ons → Secrets → label `HF_TOKEN` → toggle on for this notebook.

**Notebook settings:** Accelerator = GPU T4 ×2, Internet = On.

Save Version → Save & Run All (Commit). Nothing else to edit.

In [ ]:
# 1. Install GPU stack.
%pip -q install --upgrade unsloth unsloth_zoo
%pip -q install --no-deps trl peft accelerate bitsandbytes
%pip -q install pyyaml opencv-python-headless

In [ ]:
GITHUB_REPO     = "https://github.com/dhodyrev/handwritten-to-data.git"
HF_ADAPTER_REPO = "dkhodyriev/htr-qwen25vl-transcribe-lora"

In [ ]:
# 3. Clone repo into /kaggle/working/repo and install htr.
import os, subprocess, sys
REPO_DIR = "/kaggle/working/repo"
if not os.path.exists(REPO_DIR):
    # Private repo? Add a GH_TOKEN Kaggle Secret and uncomment:
    #   from kaggle_secrets import UserSecretsClient
    #   gh = UserSecretsClient().get_secret('GH_TOKEN')
    #   url = GITHUB_REPO.replace('https://', f'https://{gh}@')
    #   subprocess.run(['git', 'clone', url, REPO_DIR], check=True)
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], check=True)
print("cwd:", os.getcwd())

In [ ]:
# 4. Authenticate to HuggingFace using the Kaggle Secret.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("hf login OK")

In [ ]:
# 5. Data prep — all on Kaggle, no upload needed.
#    CV-split + silver-clean JSONLs are committed to the repo (small).
#    prepare_lora_data.py downloads ~8k pages from HF and materialises ~30k
#    crops. Expect 20-30 min the first time.
import pathlib
if not pathlib.Path("data/cv/train.jsonl").exists():
    !python scripts/build_cv_split.py
if not pathlib.Path("data/cv/silver_clean.jsonl").exists():
    !python scripts/clean_silver.py
!python scripts/prepare_lora_data.py --task transcribe --include-silver
!wc -l data/lora/transcribe/train.jsonl data/lora/transcribe/val.jsonl

In [ ]:
# 6. Load Qwen2.5-VL-7B (4-bit) and attach a LoRA head.
from unsloth import FastVisionModel
import torch

MODEL = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
model, processor = FastVisionModel.from_pretrained(
    MODEL, load_in_4bit=True,
    use_gradient_checkpointing="unsloth", max_seq_length=2048,
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.05,
    bias="none", random_state=42,
)
FastVisionModel.for_training(model)

In [ ]:
# 7. Load JSONL as a HF dataset (chat-format from prepare_lora_data.py).
from datasets import load_dataset
ds_train = load_dataset("json", data_files="data/lora/transcribe/train.jsonl", split="train")
ds_val   = load_dataset("json", data_files="data/lora/transcribe/val.jsonl",   split="train")
print(ds_train, ds_val)

In [ ]:
# 8. Train. Image paths in the JSONL are relative to cwd (REPO_DIR).
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=ds_train,
    eval_dataset=ds_val.select(range(min(200, len(ds_val)))),
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        num_train_epochs=1,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=1000,
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="/kaggle/working/ckpt",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        dataset_num_proc=2,
        max_seq_length=2048,
    ),
)
trainer.train()

In [ ]:
# 9. Save locally (notebook-output fallback) + push to HF Hub.
model.save_pretrained("/kaggle/working/lora_adapter")
processor.save_pretrained("/kaggle/working/lora_adapter")

model.push_to_hub(HF_ADAPTER_REPO, private=True)
processor.push_to_hub(HF_ADAPTER_REPO, private=True)
print(f"pushed adapter → https://huggingface.co/{HF_ADAPTER_REPO}")